In [1]:
import tabula
from img2pdf import convert
import numpy as np
import easyocr
import pytesseract
from paddleocr import PaddleOCR
from PIL import Image, ImageEnhance
import cv2
import re
import os
import math
import datetime
import matplotlib.pyplot as plt
import pandas as pd

c:\pessoal\projetos_individuais\ia\Iso_olhos\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
file_path_test = "C:/pessoal/projetos_individuais/ia/Iso_olhos/dados/dados_pentacam_identificados/60334_Lima_Vilma Nogueira_OD_07112024_152856_4 Maps Refr.JPG"

ocr = PaddleOCR(use_textline_orientation=True, lang='en')

reader = easyocr.Reader(['en']) 




c:\pessoal\projetos_individuais\ia\Iso_olhos\env\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\luise\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\luise\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\luise\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server

In [3]:
def load_image(image_path):
    """Load an image from the specified path using Pillow, convert to RGB and returns the image."""
    try:
        image = Image.open(image_path).convert("RGB")
        return image
    except Exception as e:
        print(f"Error loading image: {e}")
        return None

def crop_image(image, box):
    """Crop the image using the specified crop box (left, upper, right, lower)."""
    try:
        
        cropped = image.crop(box)
        return cropped
    except Exception as e:
        print(f"Error cropping image: {e}")
        return None

def extract_cornea_image(image):
    """Extract the anterior and posterior cornea images information from the given image."""

    cornea_anterior = {
        "rp": crop_image(image, box=(137, 230, 200, 247)),
        "rc": crop_image(image, box=(137, 260, 200, 280)),
        "rm": crop_image(image, box=(137, 290, 200, 310)),
        "k1": crop_image(image, box=(256, 230, 320, 247)),
        "k2": crop_image(image, box=(256, 260, 320, 280)),
        "km": crop_image(image, box=(256, 290, 320, 310)),
        "Eixo_plano": crop_image(image, box=(137, 320, 200, 340)),
        "ast": crop_image(image, box=(256, 322, 320, 339)),
        "rper": crop_image(image, box=(137, 353, 195, 368)),
        "rmin": crop_image(image, box=(256, 350, 320, 370))
    }

    cornea_posterior = {
        "rp": crop_image(image, box=(137, 415, 200, 435)),
        "rc": crop_image(image, box=(137, 445, 200, 465)),
        "rm": crop_image(image, box=(137, 475, 200, 495)),
        "k1": crop_image(image, box=(256, 415, 320, 430)),
        "k2": crop_image(image, box=(256, 445, 320, 460)),
        "km": crop_image(image, box=(256, 477, 319, 493)),
        "Eixo_plano": crop_image(image, box=(137, 505, 200, 525)),
        "ast": crop_image(image, box=(256, 507, 319, 524)),
        "rper": crop_image(image, box=(137, 539, 200, 553)),
        "rmin": crop_image(image, box=(256, 538, 319, 555))
    }

    return cornea_anterior, cornea_posterior

def extract_with_easyocr(image):
    image_np = np.array(image)

    scale_factor = 2
    scaled = cv2.resize(image_np, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_CUBIC)
    normalized = cv2.normalize(scaled, None, 0, 255, cv2.NORM_MINMAX)

    results = reader.readtext(normalized)

    outputs = []
    
    for i, (box, text, conf) in enumerate(results):
        text = text.strip() 
        text_clean = text.replace(',', '.')

        outputs.append((box, text_clean, conf))

    texts = [res[1] for res in outputs]
    full_text = " ".join(texts)

    numbers = re.findall(r'-?\d+[.,]\d+', full_text)
    numbers = [n.replace(',', '.') for n in numbers]

    return numbers if numbers else full_text.strip()

In [4]:
#testando as funções

image = load_image(file_path_test)
cornea_anterior = {}
cornea_posterior = {}
cornea_anterior, cornea_posterior = extract_cornea_image(image)

tesseract_data = {}
easyocr_data = {}

for key in cornea_anterior:
    easyocr_data[f"anterior_{key}"] = extract_with_easyocr(cornea_anterior[key])

for key in cornea_posterior:
    easyocr_data[f"posterior_{key}"] = extract_with_easyocr(cornea_posterior[key])



c:\pessoal\projetos_individuais\ia\Iso_olhos\env\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [5]:
df_original = pd.read_excel("C:/pessoal/projetos_individuais/ia/Iso_olhos/dados/Pacientes_Torica.xlsx")

exames_pentacam = os.listdir("C:/pessoal/projetos_individuais/ia/Iso_olhos/dados/dados_pentacam_identificados")


In [6]:
# trata dataframe

print(df_original.columns)

Index(['Pront', 'NOME', 'Sexo', 'OLHO OPERADO', 'K1', 'EIXO PLANO', 'K2',
       'EIXO CURVO', 'AL', 'ACD', 'LT', 'WTW', 'Refração Pré Operatória',
       'Astig Refracional pré op', 'Astig anterior topo pré op',
       'Astig posterior topo pré', 'Lente Indicada', 'Lente Implantada ',
       'Refração pós operatória', 'Astig refracional pós op ',
       'Classificação Astigmatismo '],
      dtype='object')


In [7]:
def extrair_graus(texto):
    if pd.isna(texto):
        return pd.Series([None, None, None])
    
    t = str(texto).lower().strip()
    
    # remove espaços extras e símbolos estranhos
    t = re.sub(r'[º°a]', '°', t)      # normaliza símbolo de grau
    t = re.sub(r'[+]', '+', t)
    t = re.sub(r'\s+', ' ', t)
    
    # casos "plano" e "sem rx"
    if 'sem rx' in t:
        return pd.Series([None, None, None])
    if 'pl' in t or 'plano' in t:
        t = re.sub(r'plano|pl', '0', t)
    
    # Expressão regular principal:
    padrao = re.compile(
        r'(?P<esf>[+-]?\s*\d+[.,]?\d*)'          # grau esférico
        r'(?:\s*[^\d+-]\s*|[-\s]+)'              # separador
        r'(?P<cil>[+-]?\s*\d+[.,]?\d*)?'         # grau cilíndrico (opcional)
        r'.*?(?P<eixo>\d{1,3})\s*°?'             # eixo (graus, opcional)
    )

    m = padrao.search(t)
    if m:
        esf = m.group('esf')
        cil = m.group('cil')
        eixo = m.group('eixo')
    else:
        # Tenta pegar só um número (ex: "-1,5")
        numeros = re.findall(r'[+-]?\d+[.,]?\d*', t)
        if len(numeros) == 1:
            esf, cil, eixo = numeros[0], None, None
        else:
            esf = cil = eixo = None

    # Converte vírgulas e tira espaços
    def conv(v):
        if v is None: return None
        v = v.replace(',', '.').strip()
        try:
            return float(v)
        except ValueError:
            return None

    esf = conv(esf)
    cil = conv(cil)
    eixo = int(eixo) if eixo and eixo.isdigit() else None

    return pd.Series([esf, cil, eixo])

# --- aplica no DataFrame ---


In [8]:
def extrair_lente(texto):
    if pd.isna(texto):
        return pd.Series([None, None, None])
    
    t = str(texto).strip()
    t = re.sub(r'[º°]', '°', t)  # normaliza símbolo de grau
    t = re.sub(r'\s+', ' ', t)
    
    # Expressão regular robusta:
    padrao = re.compile(
        r'(?P<topo>T\d*)?'                # parte inicial: T3, T4 etc. (opcional)
        r'\s*\+?(?P<valor>\d+[.,]?\d*)'   # valor numérico (pode ter +)
        r'\s*[Dd]?'                       # letra D opcional
        r'\s*[AaÁÀ]?\s*'                  # separador "a" ou "à"
        r'(?P<eixo>\d{1,3})?'             # eixo numérico
        r'\s*°?'                          # símbolo de grau opcional
    )
    
    m = padrao.search(t)
    if not m:
        return pd.Series([None, None, None])
    
    topo = m.group('topo')
    valor = m.group('valor')
    eixo = m.group('eixo')
    
    # Conversões
    topo = topo.strip() if topo else None
    if valor:
        valor = float(valor.replace(',', '.'))
    if eixo:
        eixo = int(eixo)
    
    return pd.Series([topo, valor, eixo])

In [9]:
from unidecode import unidecode

def limpar_tipo(texto):
    if pd.isna(texto):
        return None
    
    t = str(texto).strip().lower()
    t = unidecode(t)  # remove acentos
    t = ' '.join(t.split())  # remove espaços extras
    
    # dicionário de correções
    substituicoes = {
        'regulr simetrico obliquo': 'regular simetrico obliquo',
        'regular simetrico obliquo': 'regular simetrico obliquo',
        'regular assimetrico obliquo': 'regular assimetrico obliquo',
        'regular simetrico a favor da regra': 'regular simetrico a favor da regra',
        'regular simetrico contra a regra': 'regular simetrico contra a regra',
        'regular assimetrico a favor da regra': 'regular assimetrico a favor da regra',
        'regular assimetrico contra a regra': 'regular assimetrico contra a regra',
        'regular simetrico obliquo': 'regular simetrico obliquo',
        'regular assimetrico obliquo': 'regular assimetrico obliquo',
        'irregular': 'irregular'
    }
    
    return substituicoes.get(t, t)

In [10]:
df_original['Classificação Astigmatismo '].unique()

array(['Regular assimétrico contra a regra',
       'Regular simétrico a favor da regra',
       'Regular simétrico contra a regra',
       'Regular simetrico contra a regra ',
       'Regular assimétrico a favor da regra',
       'Regular assimetrico contra a regra', nan,
       'Regular simétrico obliquo ',
       'Regular assimetrico a favor da regra ',
       'Regular simetrico a favor da regra',
       'Regular simetrico contra a regra', 'Regular assimétrico obliquo ',
       'Irregular', 'Regular simétrico obliquo',
       'Regular assimétrico obliquo', 'Regular assimétrico oblíquo',
       'Regular simétrico oblíquo', 'Regular simétrico oblíquo ',
       'Regulr simétrico oblíquo'], dtype=object)

In [11]:
df_original[['esferico_pre', 'cilindrico_pre', 'eixo_pre']] = df_original['Refração Pré Operatória'].apply(extrair_graus)
df_original[['esferico_pos', 'cilindrico_pos', 'eixo_pos']] = df_original['Refração pós operatória'].apply(extrair_graus)
df_original[['topografia', 'valor', 'eixo']] = df_original['Lente Implantada '].apply(extrair_lente)
df_original['tipo_padronizado'] = df_original['Classificação Astigmatismo '].apply(limpar_tipo)
dummies = pd.get_dummies(df_original['tipo_padronizado'], prefix='tipo', dtype=int)
df_original = pd.concat([df_original, dummies], axis=1)
df_original['OLHO OPERADO'] = df_original['OLHO OPERADO'].replace({
    'Direito': 'OD',
    'Esquerdo': 'OS'
})
df_original['Sexo'] = df_original['Sexo'].replace({
    'Masculino': 'M',
    'Feminino': 'F'
})

df_original.columns = (
    df_original.columns
      .str.strip()          # remove espaços nas extremidades
      .str.lower()          # minúsculas
      .str.replace(r'\s+', '_', regex=True)  # espaços por underscore
      .str.replace(r'[^\w_]', '', regex=True) # remove caracteres especiais
)


colunas_excluir = [
    'refração_pré_operatória',
    'refração_pós_operatória',
    'lente_implantada',
    'lente_indicada'
]

df_original = df_original.drop(columns=colunas_excluir)

In [14]:
def find_exam_file(prontuario, olho, folder_path):
    """
    Procura o arquivo do exame correspondente ao prontuário e olho.
    Retorna o caminho completo do arquivo.
    """
    for filename in os.listdir(folder_path):
        if filename.startswith(str(prontuario)) and f"_{olho}_" in filename:
            return os.path.join(folder_path, filename)
    return None


def process_exam(file_path):
    """
    Executa o processamento da imagem e retorna um dicionário com os dados extraídos.
    """
    image = load_image(file_path)
    cornea_anterior, cornea_posterior = extract_cornea_image(image)

    easyocr_data = {}

    for key in cornea_anterior:
        easyocr_data[f"anterior_{key}"] = extract_with_easyocr(cornea_anterior[key])

    for key in cornea_posterior:
        easyocr_data[f"posterior_{key}"] = extract_with_easyocr(cornea_posterior[key])

    cleaned_data = {}
    for k, v in easyocr_data.items():
        # Se for lista do tipo ['7.73']
        if isinstance(v, list) and len(v) == 1:
            item = v[0]
        else:
            item = v
        
        # Remove possíveis caracteres não numéricos e converte pra número se possível
        try:
            item = float(str(item).replace(',', '.'))
        except ValueError:
            pass  # mantém como string se não for número
        
        cleaned_data[k] = item

    return cleaned_data

In [15]:
print(df_original.columns)

Index(['pront', 'nome', 'sexo', 'olho_operado', 'k1', 'eixo_plano', 'k2',
       'eixo_curvo', 'al', 'acd', 'lt', 'wtw', 'astig_refracional_pré_op',
       'astig_anterior_topo_pré_op', 'astig_posterior_topo_pré',
       'astig_refracional_pós_op', 'classificação_astigmatismo',
       'esferico_pre', 'cilindrico_pre', 'eixo_pre', 'esferico_pos',
       'cilindrico_pos', 'eixo_pos', 'topografia', 'valor', 'eixo',
       'tipo_padronizado', 'tipo_irregular',
       'tipo_regular_assimetrico_a_favor_da_regra',
       'tipo_regular_assimetrico_contra_a_regra',
       'tipo_regular_assimetrico_obliquo',
       'tipo_regular_simetrico_a_favor_da_regra',
       'tipo_regular_simetrico_contra_a_regra',
       'tipo_regular_simetrico_obliquo'],
      dtype='object')


In [16]:
results = []

for i, row in df_original.iterrows():
    pront = row['pront']
    olho = row['olho_operado']

    # 1️⃣ Encontrar o arquivo do exame
    file_path = find_exam_file(pront, olho, "C:/pessoal/projetos_individuais/ia/Iso_olhos/dados/dados_pentacam_identificados")

    if file_path is None:
        print(f"⚠️ Exame não encontrado para {pront} ({olho})")
        results.append({})
        continue

    # 2️⃣ Processar o exame e extrair dados
    try:
        extracted_data = process_exam(file_path)
    except Exception as e:
        print(f"Erro processando {pront}: {e}")
        extracted_data = {}

    results.append(extracted_data)


# ----------------------------
# INSERIR RESULTADOS NO DATAFRAME
# ----------------------------

# Converte a lista de dicionários em um DataFrame
df_results = pd.DataFrame(results)

# Junta os resultados às colunas originais
df_original = pd.concat([df_original, df_results], axis=1)  

c:\pessoal\projetos_individuais\ia\Iso_olhos\env\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


⚠️ Exame não encontrado para 15430 (OS)
⚠️ Exame não encontrado para 3350 (OD)
⚠️ Exame não encontrado para 117479 (OD)
⚠️ Exame não encontrado para 122190 (OD)
⚠️ Exame não encontrado para 122190 (OS)
⚠️ Exame não encontrado para 7898 (OS)
⚠️ Exame não encontrado para 120362 (OS)
⚠️ Exame não encontrado para 29617 (OD)
⚠️ Exame não encontrado para 29617 (OS)
⚠️ Exame não encontrado para 107772 (OD)
⚠️ Exame não encontrado para 107772 (OS)
⚠️ Exame não encontrado para 104714 (OD)
⚠️ Exame não encontrado para 104363 (OD)
⚠️ Exame não encontrado para 103956 (OD)
⚠️ Exame não encontrado para 103956 (OS)
⚠️ Exame não encontrado para 103507 (OS)
⚠️ Exame não encontrado para 102196 (OD)
⚠️ Exame não encontrado para 102196 (OS)
⚠️ Exame não encontrado para 101388 (OD)
⚠️ Exame não encontrado para 101388 (OS)
⚠️ Exame não encontrado para 100854 (OS)
⚠️ Exame não encontrado para 100722 (OD)
⚠️ Exame não encontrado para 99610 (OD)
⚠️ Exame não encontrado para 98846 (OS)
⚠️ Exame não encontrado p

In [17]:
df_original.to_excel("C:/pessoal/projetos_individuais/ia/Iso_olhos/dados/Pacientes_Torica_com_dados_pentacam.xlsx", index=False)

In [18]:
process_exam('C:/pessoal/projetos_individuais/ia/Iso_olhos/dados/dados_pentacam_identificados/121476_Peixoto_Ademir Martins_OD_17102024_095215_4 Maps Refr.JPG')

c:\pessoal\projetos_individuais\ia\Iso_olhos\env\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'anterior_rp': 7.73,
 'anterior_rc': 7.3,
 'anterior_rm': 7.51,
 'anterior_k1': '436D',
 'anterior_k2': 46.3,
 'anterior_km': 44.9,
 'anterior_Eixo_plano': 98.4,
 'anterior_ast': '26D',
 'anterior_rper': 7.69,
 'anterior_rmin': 7.11,
 'posterior_rp': 6.24,
 'posterior_rc': 6.18,
 'posterior_rm': 6.21,
 'posterior_k1': 6.4,
 'posterior_k2': 6.5,
 'posterior_km': 6.4,
 'posterior_Eixo_plano': 0.6,
 'posterior_ast': 0.1,
 'posterior_rper': 6.59,
 'posterior_rmin': 6.05}